In [1]:
import scanpy as sc
import anndata



import importlib

import pandas as pd
import numpy as np
#import scanpy as sc
#import scycle as cc
# import scvelo as sv
#import anndata
from sklearn.decomposition import PCA

import matplotlib as mpl
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors

ModuleNotFoundError: No module named 'scanpy'

In [ ]:
#adata = sc.read_h5ad("data/SKM_human.h5ad")
adata = sc.read_h5ad("data/SKM_mouse_raw_cells2nuclei_2022-03-30.h5ad")
#adata = sc.read_h5ad("data/SKM_mouse_pp_cells2nuclei_2022-03-30.h5ad")


In [ ]:
target_gene = "FEZ2"
target_cell_type= "Adipocyte"

In [ ]:
set((adata.obs['SampleID'].str.split("_").str[0]).tolist())

In [ ]:
adata.obs

In [ ]:
adata.var_names

In [ ]:
#cell_type_mask = adata.obs["annotation_level0"] == target_cell_type
cell_type_mask = adata.obs["annotation"] == target_cell_type


In [ ]:
if target_gene not in adata.var_names:
    raise ValueError(f"Gene {target_gene} not found in the dataset.")


In [ ]:
expression_data = adata[cell_type_mask, adata.var_names == adata.var_names].X


In [ ]:
#len(adata.X.toarray())

In [ ]:
#len(expression_data.toarray())

In [ ]:
expression_data

In [ ]:
mean_expression = expression_data.mean()
mean_expression

In [ ]:
#unique_cell_types = adata.obs["annotation_level0"].unique()
unique_cell_types = adata.obs["annotation"].unique()

unique_cell_types

In [ ]:
cell_types = unique_cell_types.tolist()

In [ ]:
len(cell_types)

In [ ]:
#cell_type_counts = adata.obs["annotation_level0"].value_counts()
cell_type_counts = adata.obs["annotation"].value_counts()

total_cells = len(adata.obs)
cell_type_proportions = {cell_type: count / total_cells for cell_type, count in cell_type_counts.items()}
cell_type_proportions

In [ ]:
valid_genes = list(adata.var_names)


In [ ]:
genes = ["NT5C2"
"ALDOA",
"BLCAP",
"FEZ2",
"SLC16A3",
"STUM",
"STUM",
"CA3",
"STIM1",
"ACIN1",
"HNRNPM",
"TPM3",
"CALM1",
"EHMT1",
"MYL2",
"RPS24",
"ENO3",
"TNNI1",
"EEF2",
"HBA2",
"MB",
"RPL13A",
"KL",
"GPX7",
"NDUFB4",
"KL",
"TAF9B",
"AK1",
"GLB1L",
"ADA",
"AK1",
"GLB1L",
"ADA",
"ERI3",
"RNF7",
"RECQL",
"CLIC4",
"AS3MT",
"CLIC6",
"MAST1",
"CLIC4",
"CCNI",
"MMP23B",
"AK1",
"CLIC4",
"CUTC",
"KLRB1",
"MICU1",
"PARK7",
"SBDS",
"MEIS2",
"ALDOA",
"CA3",
"FEZ2",
"EIF3C",
"UBE2H",
"CALM1"]  # Replace with your list of genes
genes = genes = ["NT5C2"]
# Ensure all genes are in the dataset
valid_genes = [gene for gene in genes if gene in adata.var_names]
missing_genes = set(genes) - set(valid_genes)
if missing_genes:
    print(f"Warning: The following genes are not in the dataset and will be skipped: {missing_genes}")



In [ ]:
sc.get.obs_df(adata, keys=valid_genes)

In [ ]:
sc.get.obs_df(adata, keys=['annotation'])

In [ ]:
# Initialize the results dictionary
expression_data = []

# Loop through each cell type and gene
for cell_type in cell_types:
    cell_mask = adata.obs["annotation_level0"] == cell_type
    
    for gene in valid_genes:
        mean_expression = adata[cell_mask, adata.var_names == gene].X.mean()
        expression_data.append({"Cell Type": cell_type, "Gene": gene, "Mean Expression": mean_expression})

# Convert to a Pandas DataFrame
expression_df = pd.DataFrame(expression_data)

In [ ]:
expression_df

In [ ]:
expression_df=expression_df.drop_duplicates()

In [ ]:
expression_df.to_csv("mean_expression_gene_cell_type.csv")

In [ ]:
reshaped_df = expression_df.pivot(index="Gene", columns="Cell Type", values="Mean Expression")

# Optionally, sort the index and columns for better organization
reshaped_df = reshaped_df.sort_index().sort_index(axis=1)

In [ ]:
reshaped_df["most abundant in"] = reshaped_df.idxmax(axis=1)


In [ ]:
reshaped_df["most abundant in"]

In [ ]:
reshaped_df.to_csv("results/mean_expression_cell.csv")

In [ ]:
proportion_df = pd.DataFrame.from_dict(cell_type_proportions, orient="index", columns=["Proportion"])

# Reset the index to make the cell type a column (optional)
proportion_df = proportion_df.reset_index().rename(columns={"index": "Cell Type"})
proportion_df = proportion_df.set_index("Cell Type")

In [ ]:
proportion_df.to_csv("results/proportion_cell_type.csv")